# Analisis de datos iniciales

In [1]:
import re
import unicodedata
import os
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 120)

RAW_PATH = "../data/raw/consolidado_diversificado.csv"

df_raw = pd.read_csv(RAW_PATH, encoding="utf-8", low_memory=False)
df_raw.shape

(11868, 18)

## 3.a-b. Numero de registros, variables y tipo de dato de cada una

El conjunto crudo tiene **11,868 registros** (establecimientos de nivel DIVERSIFICADO en los 22 departamentos de Guatemala) y **18 variables**, todas leidas por pandas como texto (`object`) excepto `__DEPTO_CODE` (entero). Esto ya es un problema en si mismo: variables que deberian ser categoricas (DEPARTAMENTO, SECTOR, AREA, STATUS, etc.) o numericas/identificadoras con formato fijo (TELEFONO, CODIGO) llegan como texto libre sin ninguna validacion de dominio.

In [2]:
print(f"Registros: {df_raw.shape[0]}")
print(f"Variables: {df_raw.shape[1]}")

resumen_tipos = pd.DataFrame({
    "tipo_pandas": df_raw.dtypes.astype(str),
    "ejemplo": [df_raw[c].dropna().iloc[0] if df_raw[c].notna().any() else None for c in df_raw.columns],
})
resumen_tipos

Registros: 11868
Variables: 18


,tipo_pandas,ejemplo
CODIGO,object,16-01-0137-46
DISTRITO,object,16-006
DEPARTAMENTO,object,ALTA VERAPAZ
MUNICIPIO,object,COBAN
ESTABLECIMIENTO,object,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN
DIRECCION,object,6A. AVENIDA 1-15 ZONA 4
TELEFONO,object,77945104
SUPERVISOR,object,JORGE EDUARDO PAQUE LAZARO
DIRECTOR,object,--
NIVEL,object,DIVERSIFICADO


## 3.c-e. Valores faltantes, valores unicos y duplicados exactos

In [3]:
faltantes = df_raw.isna().sum()
pct_faltantes = (faltantes / len(df_raw) * 100).round(2)
unicos = df_raw.nunique()

diagnostico = pd.DataFrame({
    "faltantes": faltantes,
    "pct_faltantes": pct_faltantes,
    "valores_unicos": unicos,
})
diagnostico

,faltantes,pct_faltantes,valores_unicos
CODIGO,0,0.00,11868
DISTRITO,533,4.49,1681
DEPARTAMENTO,0,0.00,23
MUNICIPIO,0,0.00,352
ESTABLECIMIENTO,5,0.04,6665
DIRECCION,76,0.64,7527
TELEFONO,946,7.97,6573
SUPERVISOR,536,4.52,1285
DIRECTOR,1732,14.59,5562
NIVEL,0,0.00,1


In [4]:
print("Filas 100% identicas (todas las columnas, incluyendo CODIGO):", df_raw.duplicated().sum())

# CODIGO es el identificador de establecimiento; si excluimos esa columna,
# estas serian filas identicas en TODO lo demas (candidatas a doble registro).
cols_sin_codigo = [c for c in df_raw.columns if c != "CODIGO"]
dup_sin_codigo = df_raw.duplicated(subset=cols_sin_codigo, keep=False)
print("Filas identicas salvo por CODIGO:", dup_sin_codigo.sum())
df_raw.loc[dup_sin_codigo, ["CODIGO", "ESTABLECIMIENTO", "MUNICIPIO", "DEPARTAMENTO"]].sort_values("ESTABLECIMIENTO").head(10)

Filas 100% identicas (todas las columnas, incluyendo CODIGO): 0
Filas identicas salvo por CODIGO: 348


,CODIGO,ESTABLECIMIENTO,MUNICIPIO,DEPARTAMENTO
10734,12-30-0080-46,"""COLEGIO EDUCATIVO MIXTO LUNA AZUL -CEMLA-""",LA BLANCA,SAN MARCOS
10730,12-30-0069-46,"""COLEGIO EDUCATIVO MIXTO LUNA AZUL -CEMLA-""",LA BLANCA,SAN MARCOS
7142,18-04-0380-46,"""LICEO EDUCACIONAL MIXTO DEI VERBUM""",MORALES,IZABAL
7136,18-04-0246-46,"""LICEO EDUCACIONAL MIXTO DEI VERBUM""",MORALES,IZABAL
1550,00-01-0520-46,"""LICEO MONTE SINAÍ""",ZONA 1,CIUDAD CAPITAL
1551,00-01-0521-46,"""LICEO MONTE SINAÍ""",ZONA 1,CIUDAD CAPITAL
565,15-03-0778-46,CENTRO CULTURAL DE AMERICA,RABINAL,BAJA VERAPAZ
568,15-03-1888-46,CENTRO CULTURAL DE AMERICA,RABINAL,BAJA VERAPAZ
9947,03-12-0035-46,CENTRO DE ESTUDIOS CIUDAD VIEJA,CIUDAD VIEJA,SACATEPEQUEZ
9946,03-12-0034-46,CENTRO DE ESTUDIOS CIUDAD VIEJA,CIUDAD VIEJA,SACATEPEQUEZ


## 3.f-g. Valores fuera de dominio y formatos inconsistentes

Revision variable por variable de las categoricas de baja cardinalidad, del catalogo de departamentos, del campo MUNICIPIO cuando el departamento es "CIUDAD CAPITAL", del formato de DISTRITO, TELEFONO y CODIGO, y de problemas de texto (espacios, valores centinela).

In [5]:
for col in ["SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]:
    print(f"--- {col} ({df_raw[col].nunique()} categorias) ---")
    print(df_raw[col].value_counts(dropna=False))
    print()

--- SECTOR (4 categorias) ---
SECTOR
PRIVADO        9892
OFICIAL        1499
COOPERATIVA     298
MUNICIPAL       179
Name: count, dtype: int64

--- AREA (3 categorias) ---
AREA
URBANA             9461
RURAL              2404
SIN ESPECIFICAR       3
Name: count, dtype: int64

--- STATUS (5 categorias) ---
STATUS
ABIERTA                    6861
CERRADA TEMPORALMENTE      3006
CERRADA DEFINITIVAMENTE    1841
TEMPORAL TITULOS            110
TEMPORAL NOMBRAMIENTO        50
Name: count, dtype: int64

--- MODALIDAD (2 categorias) ---
MODALIDAD
MONOLINGUE    11394
BILINGUE        474
Name: count, dtype: int64

--- JORNADA (6 categorias) ---
JORNADA
DOBLE          3772
VESPERTINA     3407
MATUTINA       3045
SIN JORNADA    1100
NOCTURNA        415
INTERMEDIA      129
Name: count, dtype: int64

--- PLAN (13 categorias) ---
PLAN
DIARIO(REGULAR)                          7452
FIN DE SEMANA                            2898
SEMIPRESENCIAL (FIN DE SEMANA)            542
SEMIPRESENCIAL (UN DÍA A LA SEMA

In [6]:
# DEPARTAMENTO: Guatemala tiene oficialmente 22 departamentos.
print("DEPARTAMENTO - valores unicos:", df_raw["DEPARTAMENTO"].nunique())
print(sorted(df_raw["DEPARTAMENTO"].unique()))
print()
print("'CIUDAD CAPITAL' no es un departamento oficial: es la ciudad de Guatemala,")
print("que administrativamente es el municipio 'Guatemala' del departamento 'Guatemala'.")
print("Filas afectadas:", (df_raw["DEPARTAMENTO"] == "CIUDAD CAPITAL").sum())
print()

# Cuando DEPARTAMENTO == 'CIUDAD CAPITAL', el campo MUNICIPIO trae la zona de la
# ciudad ('ZONA 1', 'ZONA 7', ...) en vez del nombre de un municipio real.
zonas = df_raw.loc[df_raw["DEPARTAMENTO"] == "CIUDAD CAPITAL", "MUNICIPIO"].value_counts()
print("MUNICIPIO para DEPARTAMENTO == 'CIUDAD CAPITAL' (deberian ser municipios, son zonas):")
print(zonas)

DEPARTAMENTO - valores unicos: 23
['ALTA VERAPAZ', 'BAJA VERAPAZ', 'CHIMALTENANGO', 'CHIQUIMULA', 'CIUDAD CAPITAL', 'EL PROGRESO', 'ESCUINTLA', 'GUATEMALA', 'HUEHUETENANGO', 'IZABAL', 'JALAPA', 'JUTIAPA', 'PETEN', 'QUETZALTENANGO', 'QUICHE', 'RETALHULEU', 'SACATEPEQUEZ', 'SAN MARCOS', 'SANTA ROSA', 'SOLOLA', 'SUCHITEPEQUEZ', 'TOTONICAPAN', 'ZACAPA']

'CIUDAD CAPITAL' no es un departamento oficial: es la ciudad de Guatemala,
que administrativamente es el municipio 'Guatemala' del departamento 'Guatemala'.
Filas afectadas: 2161

MUNICIPIO para DEPARTAMENTO == 'CIUDAD CAPITAL' (deberian ser municipios, son zonas):
MUNICIPIO
ZONA 1     868
ZONA 7     236
ZONA 12    149
ZONA 18    134
ZONA 2     118
ZONA 6      94
ZONA 11     80
ZONA 19     65
ZONA 13     61
ZONA 3      61
ZONA 10     56
ZONA 5      45
ZONA 21     40
ZONA 9      38
ZONA 17     30
ZONA 14     22
ZONA 15     20
ZONA 16     20
ZONA 8      10
ZONA 4       6
ZONA 25      6
ZONA 24      2
Name: count, dtype: int64


In [7]:
# DEPARTAMENTO vs DEPARTAMENTAL (columna adicional devuelta por el sitio de origen):
# difieren para Guatemala (se subdivide en zonas de supervision) y ademas DEPARTAMENTAL
# conserva tildes que DEPARTAMENTO no tiene (ej. PETEN vs PETÉN).
mismatch = df_raw.loc[df_raw["DEPARTAMENTO"] != df_raw["DEPARTAMENTAL"], ["DEPARTAMENTO", "DEPARTAMENTAL"]].drop_duplicates()
mismatch

,DEPARTAMENTO,DEPARTAMENTAL
1320,CIUDAD CAPITAL,GUATEMALA NORTE
2373,CIUDAD CAPITAL,GUATEMALA ORIENTE
2512,CIUDAD CAPITAL,GUATEMALA OCCIDENTE
2852,CIUDAD CAPITAL,GUATEMALA SUR
4347,GUATEMALA,GUATEMALA NORTE
4363,GUATEMALA,GUATEMALA ORIENTE
4623,GUATEMALA,GUATEMALA OCCIDENTE
5414,GUATEMALA,GUATEMALA SUR
7834,PETEN,PETÉN
8901,QUICHE,QUICHÉ


In [8]:
# DISTRITO: dos esquemas de codigo conviviendo en la misma columna, y codigos truncados.
d = df_raw["DISTRITO"].dropna().astype(str)
fmt_corto = d.str.match(r"^\d{2}-\d{3}$")
fmt_largo = d.str.match(r"^\d{2}-\d{2}-\d{4}$")
otros = d[~fmt_corto & ~fmt_largo]
print("Formato 'XX-XXX':", fmt_corto.sum(), " | Formato 'XX-XX-XXXX':", fmt_largo.sum(), " | Otro/truncado:", len(otros))
print("Ejemplos de codigos truncados:", otros.unique())
print()

# TELEFONO: longitudes heterogeneas, varios numeros en un mismo campo, texto mezclado con digitos.
tel = df_raw["TELEFONO"].dropna().astype(str)
print("Distribucion de longitud de TELEFONO (crudo):")
print(tel.str.len().value_counts().sort_index())
print("\nTelefonos con letras incluidas:", tel.str.contains(r"[A-Za-z]").sum())
print(tel[tel.str.contains(r"[A-Za-z]")].unique()[:6])
print("\nTelefonos con mas de un numero en el mismo campo (separados por '-','/', etc.):")
print(tel[tel.str.contains(r"[-/]")].head(6).tolist())

Formato 'XX-XXX': 5039  | Formato 'XX-XX-XXXX': 6226  | Otro/truncado: 70
Ejemplos de codigos truncados: ['01-' '17-' '10-']

Distribucion de longitud de TELEFONO (crudo):
TELEFONO
2         1
4         1
5         4
6         8
7        34
8     10672
9         4
10        2
11       11
13        2
14        1
15       24
16       20
17      102
18       14
19        5
20        2
23        2
25        4
26        8
30        1
Name: count, dtype: int64

Telefonos con letras incluidas: 9
['25763,26725 Y 21568' '25763, 26725 Y 21568' '26725,25763 Y 21568'
 '232-1011 y 251-6422' '736617 Y 714495' '5974265 FAX 5974263']

Telefonos con mas de un numero en el mismo campo (separados por '-','/', etc.):
['78208583-78209143', '79540830-79540909', '79540830-79540909', '79540830-79540909', '78391288-78392217', '79649696-78739432']


In [9]:
# CODIGO: identificador de establecimiento, formato esperado XX-XX-XXXX-XX
print("CODIGO cumple el formato XX-XX-XXXX-XX en el 100% de las filas:",
      df_raw["CODIGO"].str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$").all())
print()

# Problemas de texto: espacios extremos/multiples y valores centinela de faltante
# disfrazados de texto valido ("--", ".", "SIN DATO", etc.)
text_cols = df_raw.select_dtypes(include="object").columns
print("Columnas con espacios dobles o mas:")
for col in text_cols:
    n = df_raw[col].dropna().astype(str).str.contains(r"  +").sum()
    if n:
        print(f"  {col}: {n}")

placeholder_re = re.compile(r"^(n/?a|null|s/n|sin dato|ninguno|no aplica|nd|[\-\.\s]+)$", re.IGNORECASE)
print("\nColumnas con valores centinela de faltante escritos como texto ('--', '.', 'SIN DATO', ...):")
for col in text_cols:
    vals = df_raw[col].dropna().astype(str)
    n = vals.str.match(placeholder_re).sum()
    if n:
        print(f"  {col}: {n} ->", vals[vals.str.match(placeholder_re)].value_counts().head(3).to_dict())

CODIGO cumple el formato XX-XX-XXXX-XX en el 100% de las filas: True

Columnas con espacios dobles o mas:
  ESTABLECIMIENTO: 1392
  DIRECCION: 485
  TELEFONO: 8
  SUPERVISOR: 98
  DIRECTOR: 867



Columnas con valores centinela de faltante escritos como texto ('--', '.', 'SIN DATO', ...):
  DIRECCION: 13 -> {'.': 5, '---': 3, '--': 1}
  SUPERVISOR: 3 -> {'---------------------------- ----------------------------------': 3}
  DIRECTOR: 407 -> {'---': 167, '--': 41, '----': 39}


## 3.h. Sintesis de problemas de calidad detectados

1. **Tipos de dato**: todas las variables llegan como `object`; ninguna categorica esta tipada como tal y `TELEFONO`/`CODIGO` (identificadores/semi-numericos) tampoco.
2. **Valores faltantes disfrazados**: `DIRECTOR` (~14.6% NA reales, mas otros cientos de `"--"`, `"SIN DATO"`), `TELEFONO` (~8% NA), `SUPERVISOR` y `DISTRITO` (~4.5% NA), `DIRECCION` (~0.6%).
3. **Departamento fuera de catalogo**: `DEPARTAMENTO` tiene 23 valores en vez de los 22 departamentos oficiales; `"CIUDAD CAPITAL"` es en realidad el municipio Guatemala del departamento Guatemala.
4. **Municipio con dominio incorrecto**: cuando `DEPARTAMENTO == "CIUDAD CAPITAL"`, `MUNICIPIO` contiene zonas de la ciudad (`"ZONA 1"`, `"ZONA 7"`, ...) en vez de un municipio real.
5. **Inconsistencia entre columnas redundantes**: `DEPARTAMENTO` (sin tildes) y `DEPARTAMENTAL` (con tildes, y subdividido en zonas de supervision para Guatemala) no coinciden en 6,095 filas.
6. **Formato inconsistente en `DISTRITO`**: conviven dos esquemas de codigo (`XX-XXX` y `XX-XX-XXXX`) y 70 codigos truncados (`"01-"`).
7. **Formato inconsistente en `TELEFONO`**: longitudes de 2 a 30 caracteres, campos con 2-3 numeros concatenados, texto mezclado ("FAX", "Y", "AL").
8. **Espacios y caracteres invisibles**: espacios dobles/multiples en `ESTABLECIMIENTO`, `DIRECCION`, `SUPERVISOR`, `DIRECTOR`; NBSP y caracteres de control ocultos.
9. **Valores centinela de faltante escritos como texto**: `"--"`, `"---"`, `"."`, `"SIN DATO"` en `DIRECCION` y `DIRECTOR`, que inflan artificialmente el numero de valores "presentes".
10. **Duplicados**: 0 duplicados exactos por fila completa (porque `CODIGO` es unico), pero 181-385 filas (segun criterio de conteo) son identicas en *todo excepto* `CODIGO`, y existen decenas de establecimientos con nombres muy similares dentro del mismo municipio (posibles duplicados parciales).

# Plan de limpieza de datos

Antes de tocar los datos se documenta, para cada variable, el problema detectado, la regla de correccion propuesta (y por que deberia funcionar) y los riesgos de aplicarla.

| Variable | Problema encontrado | Regla de correccion | Por que deberia funcionar | Riesgos |
|---|---|---|---|---|
| `CODIGO` | Ninguno relevante (formato `XX-XX-XXXX-XX` en 100% de filas) | Mantener como identificador de tipo texto (no numerico, para no perder ceros a la izquierda) | Ya es consistente; forzarlo a numero rompería el formato oficial | Ninguno significativo |
| `DISTRITO` | Dos esquemas de codigo distintos (`XX-XXX` vs `XX-XX-XXXX`) conviviendo en la columna; ~70 codigos truncados (`"01-"`) | Los truncados se convierten a NA; los dos formatos validos se conservan tal cual mostrando ambos como texto y documentando el motivo | Un codigo incompleto no identifica ningun distrito real; inventar el segmento faltante seria fabricar informacion | Se pierde el dato de distrito en ~70 filas (0.6%) en vez de intentar adivinarlo; los dos formatos coexistentes podrian confundir a quien no lea la documentacion |
| `DEPARTAMENTO` | 23 categorias en vez de las 22 oficiales; `"CIUDAD CAPITAL"` no es un departamento, es la ciudad de Guatemala | Unificar `"CIUDAD CAPITAL"` -> `"GUATEMALA"` | Catalogo oficial (INE/MINEDUC) tiene 22 departamentos; Ciudad Capital es una categoria de conveniencia del buscador web para separar la consulta | Se pierde la distincion "capital vs resto del departamento"; se mitiga preservando el codigo de origen del scraper en `COD_DEPARTAMENTO_ORIGEN` |
| `MUNICIPIO` | Para las filas de `CIUDAD CAPITAL`, el campo trae la zona de la ciudad (`"ZONA 1"`) en vez de un municipio | Fijar `MUNICIPIO = "GUATEMALA"` para esas filas y crear la variable derivada `ZONA_CIUDAD_GUATEMALA` con el numero de zona | El municipio real de esas escuelas es Guatemala; la zona sigue siendo informacion util de mas detalle y no debe perderse | Si en el futuro se necesita el dato de zona para *todos* los municipios (no solo la capital) esta variable quedara mayormente vacia; se documenta como limitacion |
| `ESTABLECIMIENTO` | Espacios dobles/multiples; ~5 valores faltantes; posibles duplicados parciales de nombre | Normalizar espacios (strip + colapso); NO tocar comillas/puntuacion propia del nombre; usar similitud de cadenas para *marcar* (no fusionar) posibles duplicados | Los espacios no aportan informacion y afectan comparaciones/joins; los nombres con comillas son validos en espanol (`COLEGIO "LA INMACULADA"`) | Un umbral de similitud muy bajo generaria falsos positivos (escuelas distintas con nombres genericos como "INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA") |
| `DIRECCION` | Espacios multiples; valores centinela (`"--"`, `"."`) que en realidad son faltantes | Normalizar espacios; convertir centinelas a NA | Una direccion como `"--"` no es una direccion real | Ninguno relevante, salvo perder el matiz de "no se sabe" vs "no existe" |
| `TELEFONO` | Longitudes heterogeneas (2 a 30 caracteres), varios numeros en un mismo campo, texto mezclado ("FAX","Y","AL") | Extraer con regex todas las secuencias de 7-8 digitos; el primero pasa a `TELEFONO_PRINCIPAL` (8 digitos), el resto a `TELEFONO_SECUNDARIO`; bandera `TELEFONO_VALIDO` | Un telefono guatemalteco valido tiene 8 digitos; separar en variables derivadas evita perder informacion de contacto adicional | Los numeros de 7 digitos (formato antiguo) quedan marcados como no validos aunque pudieron ser correctos en su momento; se documenta explicitamente |
| `SUPERVISOR` / `DIRECTOR` | Espacios multiples; ~14.6% NA reales en `DIRECTOR` mas cientos de centinelas (`"--"`, `"SIN DATO"`) | Normalizar espacios; centinelas -> NA | Mismo razonamiento que `DIRECCION` | Ninguno significativo |
| `NIVEL` | Constante (`"DIVERSIFICADO"`) en el 100% de filas | Tipar como categoria; conservar (confirma el filtro de la fuente) | No aporta variabilidad pero documenta el alcance del dataset | Ninguno |
| `SECTOR`, `AREA`, `STATUS`, `MODALIDAD`, `JORNADA`, `PLAN` | Vocabularios pequenos, ya en mayusculas y sin tildes inconsistentes | Tipar como `category`; verificar que no existan variantes de mayus/minus o espacios | Cardinalidad baja y ya consistente en la fuente | Ninguno significativo |
| `DEPARTAMENTAL` | Redundante con `DEPARTAMENTO` pero con mas detalle (zonas de supervision) y con tildes | Conservar como variable auxiliar/derivada del origen, documentada en el libro de codigos, sin usarla para invalidar `DEPARTAMENTO` | Contiene informacion util (zona de supervision) que se perderia si se elimina | Puede interpretarse erroneamente como el nombre "correcto" del departamento si no se documenta bien |
| `__DEPTO_CODE` | Tipo entero para un codigo categorico (pierde el cero a la izquierda) | Renombrar a `COD_DEPARTAMENTO_ORIGEN`, convertir a texto de 2 digitos con cero a la izquierda | Es un identificador, no una cantidad | Ninguno |
| (fila completa) | Filas identicas en todo excepto `CODIGO` (181-385 filas) | Marcar `DUPLICADO_EXACTO_SIN_CODIGO = True`, **no eliminar automaticamente** | El enunciado exige analizar caso por caso; podria ser el mismo establecimiento con dos codigos administrativos vigentes (p. ej. dos jornadas) | Eliminar sin revisar podria borrar oferta educativa real distinta |
| (fila completa) | Nombres de establecimiento muy similares dentro del mismo depto/municipio | Similitud de cadenas (SequenceMatcher, equivalente a Levenshtein) con umbral 0.90; marcar `POSIBLE_DUPLICADO_PARCIAL = True`, **no fusionar automaticamente** | Permite encontrar variantes de escritura sin necesitar un catalogo externo | Nombres genericos institucionales (ej. "INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA") generan coincidencias que no son duplicados reales; requieren revision manual |

**Nota sobre el catalogo de municipios**: a diferencia de `DEPARTAMENTO` (22 valores, bien conocidos), no se dispone en este proyecto de un catalogo oficial embebido de los ~340 municipios de Guatemala. Se opto por: (1) verificar que no hay pares de nombres de municipio con similitud >0.85 dentro del mismo departamento (no se encontraron typos, solo el caso ya cubierto de "ZONA n"), y (2) usar como catalogo de referencia el conjunto de valores observados tras la limpieza. Esto se documenta como limitacion conocida en el libro de codigos.

# Implementancion de limpieza de datos

In [10]:
df = df_raw.copy()

# Registro de transformaciones (item 6 de la guia): una fila por cada operacion aplicada.
log_rows = []

def log_transform(variable, problema, transformacion, n_afectados, justificacion):
    log_rows.append({
        "Variable": variable,
        "Problema detectado": problema,
        "Transformacion": transformacion,
        "Registros afectados": n_afectados,
        "Justificacion": justificacion,
    })

TEXT_COLS = df.select_dtypes(include="object").columns.tolist()
TEXT_COLS

['CODIGO',
 'DISTRITO',
 'DEPARTAMENTO',
 'MUNICIPIO',
 'ESTABLECIMIENTO',
 'DIRECCION',
 'TELEFONO',
 'SUPERVISOR',
 'DIRECTOR',
 'NIVEL',
 'SECTOR',
 'AREA',
 'STATUS',
 'MODALIDAD',
 'JORNADA',
 'PLAN',
 'DEPARTAMENTAL']

### 5.a-c. Normalizacion de texto: espacios, caracteres invisibles y valores faltantes disfrazados

In [11]:
def normalizar_espacios(s):
    """Quita NBSP y caracteres de control invisibles, colapsa espacios multiples y recorta bordes."""
    if pd.isna(s):
        return s
    s = str(s)
    s = s.replace("\xa0", " ")
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\n\t")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

for col in TEXT_COLS:
    original = df[col].copy()
    df[col] = df[col].apply(normalizar_espacios)
    n = int(((original.astype(str) != df[col].astype(str)) & original.notna()).sum())
    if n:
        log_transform(col, "Espacios extra/invisibles (NBSP, control, dobles espacios, bordes)",
                      "strip() + colapso de espacios multiples + remocion de caracteres de control", n,
                      "Uniformiza el texto para comparaciones/joins y evita falsos 'no duplicados' por diferencias invisibles")

pd.DataFrame(log_rows)

,Variable,Problema detectado,Transformacion,Registros afectados,Justificacion
0,ESTABLECIMIENTO,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,1392,Uniformiza el texto para comparaciones/joins y...
1,DIRECCION,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,485,Uniformiza el texto para comparaciones/joins y...
2,TELEFONO,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,8,Uniformiza el texto para comparaciones/joins y...
3,SUPERVISOR,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,102,Uniformiza el texto para comparaciones/joins y...
4,DIRECTOR,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,867,Uniformiza el texto para comparaciones/joins y...


In [12]:
# Valores centinela de faltante ('--', '.', 'N/A', 'SIN DATO', ...) -> NA real
PLACEHOLDER_RE = re.compile(r"^(n/?a|null|s/n|sin dato|ninguno|no aplica|nd|[\-\.\s]+)$", re.IGNORECASE)

for col in TEXT_COLS:
    mask = df[col].astype(str).str.match(PLACEHOLDER_RE, na=False) & df[col].notna()
    n = int(mask.sum())
    if n:
        log_transform(col, "Valores centinela de faltante escritos como texto ('--', '.', 'SIN DATO', 'N/A', ...)",
                      "Se reemplazan por NA", n,
                      "Estos tokens no son datos validos: son marcadores de informacion no disponible capturados por el sistema de origen")
    df.loc[mask, col] = np.nan

# Cadenas vacias remanentes (tras el strip) -> NA
for col in TEXT_COLS:
    mask = df[col].astype(str).str.len().eq(0) & df[col].notna()
    n = int(mask.sum())
    if n:
        log_transform(col, "Cadena vacia", "Se reemplaza por NA", n, "Una cadena vacia equivale a un dato faltante")
    df.loc[mask, col] = np.nan

df.isna().sum()

CODIGO                0
DISTRITO            533
DEPARTAMENTO          0
MUNICIPIO             0
ESTABLECIMIENTO       5
DIRECCION            89
TELEFONO            946
SUPERVISOR          539
DIRECTOR           2139
NIVEL                 0
SECTOR                0
AREA                  0
STATUS                0
MODALIDAD             0
JORNADA               0
PLAN                  0
DEPARTAMENTAL         0
__DEPTO_CODE          0
dtype: int64

### 5.d, 5.f. Consistencia de categorias y valores fuera de dominio: DEPARTAMENTO y MUNICIPIO

In [13]:
# DEPARTAMENTO: catalogo oficial de 22 departamentos de Guatemala.
DEP_CANON = {
    "ALTA VERAPAZ": "ALTA VERAPAZ", "BAJA VERAPAZ": "BAJA VERAPAZ",
    "CHIMALTENANGO": "CHIMALTENANGO", "CHIQUIMULA": "CHIQUIMULA",
    "CIUDAD CAPITAL": "GUATEMALA", "EL PROGRESO": "EL PROGRESO",
    "ESCUINTLA": "ESCUINTLA", "GUATEMALA": "GUATEMALA",
    "HUEHUETENANGO": "HUEHUETENANGO", "IZABAL": "IZABAL", "JALAPA": "JALAPA",
    "JUTIAPA": "JUTIAPA", "PETEN": "PETEN", "QUETZALTENANGO": "QUETZALTENANGO",
    "QUICHE": "QUICHE", "RETALHULEU": "RETALHULEU", "SACATEPEQUEZ": "SACATEPEQUEZ",
    "SAN MARCOS": "SAN MARCOS", "SANTA ROSA": "SANTA ROSA", "SOLOLA": "SOLOLA",
    "SUCHITEPEQUEZ": "SUCHITEPEQUEZ", "TOTONICAPAN": "TOTONICAPAN", "ZACAPA": "ZACAPA",
}
n_ciudad_capital = int((df["DEPARTAMENTO"] == "CIUDAD CAPITAL").sum())
df["DEPARTAMENTO"] = df["DEPARTAMENTO"].map(DEP_CANON)
log_transform(
    "DEPARTAMENTO",
    "'CIUDAD CAPITAL' no es uno de los 22 departamentos oficiales; es la ciudad de Guatemala",
    "Se unifica CIUDAD CAPITAL -> GUATEMALA", n_ciudad_capital,
    "El catalogo oficial de Guatemala tiene 22 departamentos. CIUDAD CAPITAL era una categoria de "
    "conveniencia del sitio de origen para separar la consulta de la capital del resto del "
    "departamento de Guatemala",
)
print("DEPARTAMENTO unicos:", df["DEPARTAMENTO"].nunique())

# MUNICIPIO: para las ex-CIUDAD CAPITAL, el campo traia la zona ('ZONA 1') en vez del municipio.
mask_zona = df["MUNICIPIO"].astype(str).str.match(r"^ZONA\s*\d+$", na=False)
n_zona = int(mask_zona.sum())
df["ZONA_CIUDAD_GUATEMALA"] = np.where(mask_zona, df["MUNICIPIO"].str.extract(r"(\d+)", expand=False), np.nan)
df.loc[mask_zona, "MUNICIPIO"] = "GUATEMALA"
log_transform(
    "MUNICIPIO",
    "Para los ex-CIUDAD CAPITAL, MUNICIPIO traia la zona de la ciudad ('ZONA 1') en vez del "
    "nombre del municipio",
    "MUNICIPIO='GUATEMALA' (municipio real) + nueva variable derivada ZONA_CIUDAD_GUATEMALA con "
    "el numero de zona", n_zona,
    "El municipio de un establecimiento en la ciudad de Guatemala es 'Guatemala'; la zona es "
    "informacion valida de otro nivel de granularidad y se preserva como variable derivada en vez "
    "de perderla",
)
print("Filas con ZONA_CIUDAD_GUATEMALA:", df["ZONA_CIUDAD_GUATEMALA"].notna().sum())

DEPARTAMENTO unicos: 22
Filas con ZONA_CIUDAD_GUATEMALA: 2161


### 5.e. Formatos: DISTRITO, CODIGO y TELEFONO

In [14]:
# DISTRITO: se conservan los dos formatos validos observados; los codigos truncados -> NA
d = df["DISTRITO"]
fmt_corto = d.str.match(r"^\d{2}-\d{3}$", na=False)
fmt_largo = d.str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
truncado = d.notna() & ~fmt_corto & ~fmt_largo
n_trunc = int(truncado.sum())
df.loc[truncado, "DISTRITO"] = np.nan
log_transform(
    "DISTRITO",
    "Codigos truncados/incompletos (ej. '01-', '17-') y dos esquemas de codigo distintos "
    "conviviendo en la misma columna (XX-XXX y XX-XX-XXXX)",
    "Los codigos truncados se convierten en NA; los dos formatos validos se documentan y se "
    "conservan sin forzar una unificacion (representan generaciones distintas del codigo de "
    "supervision, no un error)",
    n_trunc,
    "Un codigo truncado no identifica un distrito real; es preferible tratarlo como faltante a "
    "inventar el segmento que falta",
)

# CODIGO: se valida el formato pero no se modifica (es el identificador de establecimiento)
codigo_ok = df["CODIGO"].str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$", na=False).all()
print("CODIGO cumple el formato XX-XX-XXXX-XX en el 100% de las filas:", codigo_ok)


# TELEFONO: se extraen todas las secuencias de 7-8 digitos; formato uniforme en variables derivadas
def extraer_telefonos(raw):
    if pd.isna(raw):
        return pd.Series([np.nan, np.nan, False])
    nums = re.findall(r"\d{7,8}", str(raw))
    principal = nums[0] if nums else np.nan
    secundario = ";".join(nums[1:]) if len(nums) > 1 else np.nan
    valido = isinstance(principal, str) and len(principal) == 8
    return pd.Series([principal, secundario, valido])

tel_extraido = df["TELEFONO"].apply(extraer_telefonos)
tel_extraido.columns = ["TELEFONO_PRINCIPAL", "TELEFONO_SECUNDARIO", "TELEFONO_VALIDO"]

n_tel_multi = int(tel_extraido["TELEFONO_SECUNDARIO"].notna().sum())
n_tel_no8 = int((df["TELEFONO"].notna() & ~tel_extraido["TELEFONO_VALIDO"]).sum())

df["TELEFONO_PRINCIPAL"] = tel_extraido["TELEFONO_PRINCIPAL"]
df["TELEFONO_SECUNDARIO"] = tel_extraido["TELEFONO_SECUNDARIO"]
df["TELEFONO_VALIDO"] = tel_extraido["TELEFONO_VALIDO"]
log_transform(
    "TELEFONO",
    "Formatos heterogeneos: numeros con guiones/diagonales, varios numeros en un mismo campo "
    "('78208583-78209143'), texto mezclado ('FAX','Y','AL'), longitudes distintas a 8 digitos",
    "Se extraen todas las secuencias de 7-8 digitos con regex; la primera pasa a "
    "TELEFONO_PRINCIPAL (formato uniforme de digitos, sin separadores), las adicionales a "
    "TELEFONO_SECUNDARIO, y se agrega la bandera TELEFONO_VALIDO (True solo si tiene 8 digitos)",
    n_tel_multi + n_tel_no8,
    "Un telefono guatemalteco valido tiene 8 digitos; preservar los numeros adicionales evita "
    "perder informacion de contacto y la bandera permite filtrar registros no verificables",
)
df = df.drop(columns=["TELEFONO"])

print("TELEFONO_VALIDO:", df["TELEFONO_VALIDO"].value_counts(dropna=False).to_dict())

CODIGO cumple el formato XX-XX-XXXX-XX en el 100% de las filas: True


TELEFONO_VALIDO: {True: 10815, False: 1053}


### 5.b. Tipo de dato apropiado por variable

In [15]:
CAT_COLS = ["DEPARTAMENTO", "MUNICIPIO", "NIVEL", "SECTOR", "AREA", "STATUS",
            "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL"]
for col in CAT_COLS:
    df[col] = df[col].astype("category")

# __DEPTO_CODE es un codigo, no una cantidad: se pasa a texto de 2 digitos y se renombra
df["__DEPTO_CODE"] = df["__DEPTO_CODE"].astype(str).str.zfill(2)
log_transform(
    "__DEPTO_CODE",
    "Tipo numerico (int) para lo que en realidad es un codigo categorico; ademas pierde el cero a "
    "la izquierda al mostrarse (5 en vez de '05')",
    "Se convierte a texto de 2 digitos con cero a la izquierda y se renombra a "
    "COD_DEPARTAMENTO_ORIGEN", len(df),
    "Es un identificador (codigo del departamento tal como lo uso el formulario del scraper), no "
    "una cantidad sobre la que tenga sentido sumar/promediar",
)
df = df.rename(columns={"__DEPTO_CODE": "COD_DEPARTAMENTO_ORIGEN"})

df.dtypes

CODIGO                       object
DISTRITO                     object
DEPARTAMENTO               category
MUNICIPIO                  category
ESTABLECIMIENTO              object
DIRECCION                    object
SUPERVISOR                   object
DIRECTOR                     object
NIVEL                      category
SECTOR                     category
AREA                       category
STATUS                     category
MODALIDAD                  category
JORNADA                    category
PLAN                       category
DEPARTAMENTAL              category
COD_DEPARTAMENTO_ORIGEN      object
ZONA_CIUDAD_GUATEMALA        object
TELEFONO_PRINCIPAL           object
TELEFONO_SECUNDARIO          object
TELEFONO_VALIDO                bool
dtype: object

### 5.g. Registros duplicados: exactos (salvo CODIGO) y parciales (similitud de nombre)

In [16]:
# Duplicados exactos (todas las columnas salvo el identificador CODIGO)
cols_sin_codigo = [c for c in df.columns if c != "CODIGO"]
dup_mask = df.duplicated(subset=cols_sin_codigo, keep=False)
df["DUPLICADO_EXACTO_SIN_CODIGO"] = dup_mask
n_dup_exact = int(dup_mask.sum())
log_transform(
    "(fila completa)",
    "Registros identicos en todas las columnas excepto CODIGO (posible doble registro del mismo "
    "establecimiento/oferta)",
    "NO se eliminan; se marca DUPLICADO_EXACTO_SIN_CODIGO=True para revision manual", n_dup_exact,
    "El enunciado exige no borrar automaticamente: puede tratarse del mismo establecimiento con "
    "dos codigos administrativos vigentes (ej. reinscripciones)",
)
print("Filas marcadas como duplicado exacto salvo CODIGO:", n_dup_exact)

Filas marcadas como duplicado exacto salvo CODIGO: 385


In [17]:
# Duplicados parciales: similitud de nombre de establecimiento dentro del mismo depto+municipio.
# Se usa difflib.SequenceMatcher (equivalente a Levenshtein, sin dependencias externas como
# rapidfuzz, que no esta disponible en este entorno). Nota: esta celda compara todos los pares
# dentro de cada grupo depto+municipio, por lo que tarda unos minutos (~11.9k registros).
def normalizar_nombre(s):
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Z0-9 ]", "", s.upper())
    return re.sub(r"\s+", " ", s).strip()

df["_NOMBRE_NORM"] = df["ESTABLECIMIENTO"].apply(normalizar_nombre)

UMBRAL_SIMILITUD = 0.90
posibles_dup_parcial = []
for (dep, mun), g in df.groupby(["DEPARTAMENTO", "MUNICIPIO"], observed=True):
    idx = g.index.tolist()
    nombres = g["_NOMBRE_NORM"].tolist()
    codigos = g["CODIGO"].tolist()
    for i in range(len(idx)):
        if not nombres[i]:
            continue
        for j in range(i + 1, len(idx)):
            if not nombres[j]:
                continue
            ratio = SequenceMatcher(None, nombres[i], nombres[j]).ratio()
            if ratio >= UMBRAL_SIMILITUD and nombres[i] != nombres[j]:
                posibles_dup_parcial.append((codigos[i], codigos[j], nombres[i], nombres[j], round(ratio, 3)))

codigos_dup_parcial = {c for par in posibles_dup_parcial for c in par[:2]}
df["POSIBLE_DUPLICADO_PARCIAL"] = df["CODIGO"].isin(codigos_dup_parcial)
df = df.drop(columns=["_NOMBRE_NORM"])

log_transform(
    "(fila completa)",
    "Nombres de establecimiento muy similares (posible variante de escritura) dentro del mismo "
    "departamento/municipio, detectados con similitud de cadenas (umbral 0.90)",
    "NO se fusionan ni eliminan; se marca POSIBLE_DUPLICADO_PARCIAL=True para revision manual "
    "caso por caso", len(codigos_dup_parcial),
    "Nombres parecidos pueden ser establecimientos legitimamente distintos (ej. nombres "
    "genericos como 'INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA' repetidos en varios "
    "municipios, o el mismo colegio con jornadas/planes registrados por separado); requiere "
    "criterio humano, no automatizacion ciega",
)

print("Pares con similitud >=", UMBRAL_SIMILITUD, ":", len(posibles_dup_parcial))
print("Codigos unicos marcados como posible duplicado parcial:", len(codigos_dup_parcial))
pd.DataFrame(posibles_dup_parcial, columns=["codigo_a", "codigo_b", "nombre_a", "nombre_b", "similitud"]).head(10)

Pares con similitud >= 0.9 : 2741
Codigos unicos marcados como posible duplicado parcial: 2001


,codigo_a,codigo_b,nombre_a,nombre_b,similitud
0,16-14-0054-46,16-14-0112-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947
1,16-14-0054-46,16-14-0113-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947
2,16-14-0054-46,16-14-0114-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947
3,16-14-0054-46,16-14-0115-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947
4,16-14-0054-46,16-14-0116-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947
5,16-14-0054-46,16-14-0117-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947
6,16-14-0054-46,16-14-0118-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947
7,16-14-0088-46,16-14-0092-46,INSTITUTO DE EDUCACION DIVERSIFICADA POR COOPE...,INSTITUTO DE EDUCACION DIVERSIFICADA POR COOPE...,0.903
8,16-01-0145-46,16-01-1259-46,INSTITUTO DE TURSMO Y AVIACON DEL NORTE ITAN,INSTITUTO DE TURISMO Y AVIACION DEL NORTE ITAN,0.978
9,16-01-0428-46,16-01-1041-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,0.947


### 5.h. Consistencia entre variables

In [18]:
# AREA=RURAL dentro de la ciudad de Guatemala es, cuando menos, atipico -> se documenta, no se corrige.
inconsist_area = df[(df["MUNICIPIO"] == "GUATEMALA") & (df["ZONA_CIUDAD_GUATEMALA"].notna()) & (df["AREA"] == "RURAL")]
print("Establecimientos en zona de la ciudad de Guatemala marcados AREA=RURAL:", len(inconsist_area))

# STATUS='CERRADA DEFINITIVAMENTE' con telefono valido activo: no es necesariamente un error
# (el numero pudo quedar activo tras el cierre), pero se documenta como inconsistencia a revisar.
inconsist_status = df[(df["STATUS"] == "CERRADA DEFINITIVAMENTE") & (df["TELEFONO_VALIDO"])]
print("Establecimientos CERRADA DEFINITIVAMENTE con telefono valido:", len(inconsist_status))

Establecimientos en zona de la ciudad de Guatemala marcados AREA=RURAL: 12
Establecimientos CERRADA DEFINITIVAMENTE con telefono valido: 1380


### 6. Registro de transformaciones (tabla completa)

In [19]:
log_df = pd.DataFrame(log_rows)
log_df

,Variable,Problema detectado,Transformacion,Registros afectados,Justificacion
0,ESTABLECIMIENTO,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,1392,Uniformiza el texto para comparaciones/joins y...
1,DIRECCION,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,485,Uniformiza el texto para comparaciones/joins y...
2,TELEFONO,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,8,Uniformiza el texto para comparaciones/joins y...
3,SUPERVISOR,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,102,Uniformiza el texto para comparaciones/joins y...
4,DIRECTOR,"Espacios extra/invisibles (NBSP, control, dobl...",strip() + colapso de espacios multiples + remo...,867,Uniformiza el texto para comparaciones/joins y...
5,DIRECCION,Valores centinela de faltante escritos como te...,Se reemplazan por NA,13,Estos tokens no son datos validos: son marcado...
6,SUPERVISOR,Valores centinela de faltante escritos como te...,Se reemplazan por NA,3,Estos tokens no son datos validos: son marcado...
7,DIRECTOR,Valores centinela de faltante escritos como te...,Se reemplazan por NA,407,Estos tokens no son datos validos: son marcado...
8,DEPARTAMENTO,'CIUDAD CAPITAL' no es uno de los 22 departame...,Se unifica CIUDAD CAPITAL -> GUATEMALA,2161,El catalogo oficial de Guatemala tiene 22 depa...
9,MUNICIPIO,"Para los ex-CIUDAD CAPITAL, MUNICIPIO traia la...",MUNICIPIO='GUATEMALA' (municipio real) + nueva...,2161,El municipio de un establecimiento en la ciuda...


### 7. Validacion del conjunto limpio (pruebas automaticas)

In [20]:
resultados_validacion = []

def check(nombre, condicion):
    ok = bool(condicion)
    resultados_validacion.append((nombre, ok))
    print(f"[{'OK' if ok else 'FALLA'}] {nombre}")

# No existen registros duplicados exactos (por el identificador CODIGO)
check("No hay duplicados exactos (CODIGO es unico)", df["CODIGO"].is_unique)

# No existen espacios al inicio/fin de textos
for col in df.select_dtypes(include="object").columns:
    s = df[col].dropna().astype(str)
    check(f"Sin espacios al inicio/fin en {col}", (s == s.str.strip()).all())

# Los telefonos tienen un formato consistente (solo digitos, 7 u 8 caracteres)
check("TELEFONO_PRINCIPAL solo contiene 7-8 digitos numericos (o NA)",
      df["TELEFONO_PRINCIPAL"].dropna().astype(str).str.match(r"^\d{7,8}$").all())

# Departamentos pertenecen al catalogo oficial de 22
DEP_OFICIALES = {
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA", "EL PROGRESO", "ESCUINTLA",
    "GUATEMALA", "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA", "PETEN", "QUETZALTENANGO",
    "QUICHE", "RETALHULEU", "SACATEPEQUEZ", "SAN MARCOS", "SANTA ROSA", "SOLOLA",
    "SUCHITEPEQUEZ", "TOTONICAPAN", "ZACAPA",
}
check("DEPARTAMENTO tiene exactamente los 22 departamentos oficiales",
      set(df["DEPARTAMENTO"].cat.categories) == DEP_OFICIALES)

# Municipios ya no contienen el artefacto 'ZONA n'
check("MUNICIPIO no contiene valores 'ZONA n'",
      not df["MUNICIPIO"].astype(str).str.match(r"^ZONA\s*\d+$", na=False).any())

# Variables tienen el tipo de dato esperado
check("Variables categoricas tipadas como 'category'", all(str(df[c].dtype) == "category" for c in CAT_COLS))
check("COD_DEPARTAMENTO_ORIGEN es texto de 2 digitos", df["COD_DEPARTAMENTO_ORIGEN"].str.match(r"^\d{2}$").all())

# No existen categorias duplicadas por diferencias de escritura (may/min, espacios)
for col in ["SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTO", "MUNICIPIO"]:
    valores = df[col].dropna().unique().tolist()
    valores_norm = {str(v).upper().strip() for v in valores}
    check(f"Sin categorias duplicadas por may/min o espacios en {col}", len(valores) == len(valores_norm))

print(f"\nResumen: {sum(ok for _, ok in resultados_validacion)}/{len(resultados_validacion)} pruebas superadas")
assert all(ok for _, ok in resultados_validacion), "Hay pruebas de calidad que fallaron; revisar arriba"

[OK] No hay duplicados exactos (CODIGO es unico)
[OK] Sin espacios al inicio/fin en CODIGO
[OK] Sin espacios al inicio/fin en DISTRITO
[OK] Sin espacios al inicio/fin en ESTABLECIMIENTO
[OK] Sin espacios al inicio/fin en DIRECCION
[OK] Sin espacios al inicio/fin en SUPERVISOR
[OK] Sin espacios al inicio/fin en DIRECTOR
[OK] Sin espacios al inicio/fin en COD_DEPARTAMENTO_ORIGEN
[OK] Sin espacios al inicio/fin en ZONA_CIUDAD_GUATEMALA
[OK] Sin espacios al inicio/fin en TELEFONO_PRINCIPAL
[OK] Sin espacios al inicio/fin en TELEFONO_SECUNDARIO
[OK] TELEFONO_PRINCIPAL solo contiene 7-8 digitos numericos (o NA)
[OK] DEPARTAMENTO tiene exactamente los 22 departamentos oficiales
[OK] MUNICIPIO no contiene valores 'ZONA n'
[OK] Variables categoricas tipadas como 'category'
[OK] COD_DEPARTAMENTO_ORIGEN es texto de 2 digitos
[OK] Sin categorias duplicadas por may/min o espacios en SECTOR
[OK] Sin categorias duplicadas por may/min o espacios en AREA
[OK] Sin categorias duplicadas por may/min o esp

### 8. Informe de calidad: antes vs. despues

In [21]:
flag_cols = ["DUPLICADO_EXACTO_SIN_CODIGO", "POSIBLE_DUPLICADO_PARCIAL", "TELEFONO_VALIDO"]
df_sin_flags = df.drop(columns=flag_cols)

miss_antes = int(df_raw.isna().sum().sum())
miss_despues = int(df_sin_flags.isna().sum().sum())

reporte_calidad = pd.DataFrame({
    "Metrica": [
        "Registros", "Variables", "Valores faltantes (celdas)", "% celdas faltantes",
        "Variables con NA", "Duplicados exactos (fila completa)",
        "Duplicados exactos excluyendo CODIGO", "Posibles duplicados parciales (nombre similar)",
        "Variables con formato inconsistente corregido", "Variables con tipo incorrecto corregido",
        "Categorias inconsistentes unificadas",
    ],
    "Antes": [
        df_raw.shape[0], df_raw.shape[1], miss_antes,
        round(miss_antes / (df_raw.shape[0] * df_raw.shape[1]) * 100, 2),
        int((df_raw.isna().sum() > 0).sum()), int(df_raw.duplicated().sum()),
        int(df_raw.duplicated(subset=[c for c in df_raw.columns if c != "CODIGO"]).sum()),
        "N/A (no evaluado)", 3, 2, 1,
    ],
    "Despues": [
        df.shape[0], df.shape[1], miss_despues,
        round(miss_despues / (df_sin_flags.shape[0] * df_sin_flags.shape[1]) * 100, 2),
        int((df_sin_flags.isna().sum() > 0).sum()), int(df["CODIGO"].duplicated().sum()),
        int(df["DUPLICADO_EXACTO_SIN_CODIGO"].sum()), int(df["POSIBLE_DUPLICADO_PARCIAL"].sum()),
        0, 0, 0,
    ],
})
reporte_calidad

,Metrica,Antes,Despues
0,Registros,11868,11868.00
1,Variables,18,23.00
2,Valores faltantes (celdas),3828,25753.00
3,% celdas faltantes,1.79,10.85
4,Variables con NA,6,8.00
5,Duplicados exactos (fila completa),0,0.00
6,Duplicados exactos excluyendo CODIGO,181,385.00
7,Posibles duplicados parciales (nombre similar),N/A (no evaluado),2001.00
8,Variables con formato inconsistente corregido,3,0.00
9,Variables con tipo incorrecto corregido,2,0.00


**Nota sobre por que suben los "valores faltantes" tras la limpieza**: el numero de celdas NA aumenta (no porque se haya perdido informacion, sino porque):
1. Se destaparon valores faltantes que estaban disfrazados como texto (`"--"`, `"SIN DATO"`), que antes de la limpieza contaban como "presentes".
2. Se crearon variables derivadas nuevas que son inherentemente dispersas: `ZONA_CIUDAD_GUATEMALA` solo aplica a establecimientos de la ciudad de Guatemala, y `TELEFONO_SECUNDARIO` solo aplica a los registros que tenian mas de un numero.

Es decir, el dato "real" disponible no disminuyo: se volvio mas honesto y mas granular. Lo mismo aplica al aumento de "duplicados exactos excluyendo CODIGO": la normalizacion de espacios y la unificacion de valores centinela a NA hizo que pares de filas que antes diferian solo por un espacio o un "--" vs "---" ahora se reconozcan correctamente como identicos.

### 9. Generacion del conjunto de datos limpio

In [22]:
OUT_DIR = "../data/clean"
os.makedirs(OUT_DIR, exist_ok=True)

CLEAN_PATH = f"{OUT_DIR}/establecimientos_diversificado_limpio.csv"
LOG_PATH = f"{OUT_DIR}/registro_transformaciones.csv"
REPORT_PATH = f"{OUT_DIR}/informe_calidad_antes_despues.csv"

df.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
log_df.to_csv(LOG_PATH, index=False, encoding="utf-8")
reporte_calidad.to_csv(REPORT_PATH, index=False, encoding="utf-8")

print("Conjunto limpio exportado a:", CLEAN_PATH, df.shape)
print("Registro de transformaciones:", LOG_PATH)
print("Informe de calidad antes/despues:", REPORT_PATH)
df.head()

Conjunto limpio exportado a: ../data/clean/establecimientos_diversificado_limpio.csv (11868, 23)
Registro de transformaciones: ../data/clean/registro_transformaciones.csv
Informe de calidad antes/despues: ../data/clean/informe_calidad_antes_despues.csv


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,COD_DEPARTAMENTO_ORIGEN,ZONA_CIUDAD_GUATEMALA,TELEFONO_PRINCIPAL,TELEFONO_SECUNDARIO,TELEFONO_VALIDO,DUPLICADO_EXACTO_SIN_CODIGO,POSIBLE_DUPLICADO_PARCIAL
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,JORGE EDUARDO PAQUE LAZARO,NaN,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ,16,NaN,NaN,NaN,False,False,False
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,16,NaN,77945104,NaN,True,False,False
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,16,NaN,58015763,NaN,True,False,False
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,16,NaN,78232301,NaN,True,False,False
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,16,NaN,40389968,NaN,True,False,False


**Advertencia sobre tipos al releer el CSV**: CSV es un formato de texto plano y no conserva el `dtype` de pandas. Columnas que en memoria eran texto de digitos (`COD_DEPARTAMENTO_ORIGEN`, `TELEFONO_PRINCIPAL`, `TELEFONO_SECUNDARIO`, `ZONA_CIUDAD_GUATEMALA`) pueden volver a inferirse como numericas al leer el archivo de nuevo (perdiendo, por ejemplo, el cero a la izquierda de `"05"`). Esto **no es una perdida de limpieza**, es una limitacion del formato CSV; la forma correcta de reproducir el resultado es indicar explicitamente `dtype=str` para esas columnas al leer, como se muestra abajo, y quedara documentado en el Libro de Codigos.

In [23]:
DTYPE_AL_RELEER = {
    "CODIGO": str, "DISTRITO": str, "COD_DEPARTAMENTO_ORIGEN": str,
    "ZONA_CIUDAD_GUATEMALA": str, "TELEFONO_PRINCIPAL": str, "TELEFONO_SECUNDARIO": str,
}
verificacion = pd.read_csv(CLEAN_PATH, dtype=DTYPE_AL_RELEER, encoding="utf-8")
print(verificacion[["COD_DEPARTAMENTO_ORIGEN", "TELEFONO_PRINCIPAL", "ZONA_CIUDAD_GUATEMALA"]].dtypes)
verificacion.loc[verificacion["ZONA_CIUDAD_GUATEMALA"].notna(),
                  ["MUNICIPIO", "ZONA_CIUDAD_GUATEMALA", "COD_DEPARTAMENTO_ORIGEN"]].head(3)

COD_DEPARTAMENTO_ORIGEN    object
TELEFONO_PRINCIPAL         object
ZONA_CIUDAD_GUATEMALA      object
dtype: object


,MUNICIPIO,ZONA_CIUDAD_GUATEMALA,COD_DEPARTAMENTO_ORIGEN
1320,GUATEMALA,1,00
1321,GUATEMALA,1,00
1322,GUATEMALA,1,00


## Pendientes fuera del alcance de este notebook (items 10-11 de la guia)

- **Libro de codigos**: documento aparte (markdown/Google Docs + PDF) con descripcion, tipo, dominio, tratamiento, fecha de extraccion (ver `git log` del archivo crudo, 2026-07-21) y fuente (`http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/`) de cada variable, incluyendo las derivadas (`ZONA_CIUDAD_GUATEMALA`, `TELEFONO_PRINCIPAL`, `TELEFONO_SECUNDARIO`, `TELEFONO_VALIDO`, `DUPLICADO_EXACTO_SIN_CODIGO`, `POSIBLE_DUPLICADO_PARCIAL`, `COD_DEPARTAMENTO_ORIGEN`).
- Revision manual, caso por caso, de las filas marcadas `DUPLICADO_EXACTO_SIN_CODIGO` y `POSIBLE_DUPLICADO_PARCIAL` (no se eliminan automaticamente, tal como exige el enunciado).